# Revision Evidence Notebook
**Thesis:** Fine-Tuned YOLO11 for Ship Detection in the Istanbul Bosphorus Strait\
**Author:** Recep Ertugrul Eksi - Uskudar University, Computer Engineering (MSc)

This notebook extends the evaluation protocol of `compare_models.ipynb` for the post-defense revisions. It produces:

1. **Confidence-threshold sweep** for all five models (committee request: threshold comparisons)
2. **IoU matching-threshold sensitivity** analysis
3. **Precision-recall curves** for the fine-tuned model and the strongest vanilla baseline
4. **Regenerated figures with English labels** ("ship" instead of "gemiler")
5. **Dataset statistics** computed from the published dataset export

All evaluation parameters are identical to the original protocol: vanilla models at 640 px, fine-tuned at 1088 px, greedy IoU matching at 0.5, no class filtering.

**How to run:** Runtime > Change runtime type > select a GPU, then Runtime > Run all. Takes roughly 15-25 minutes. Outputs are saved to Google Drive under `Comparisons/revision_outputs/`.

In [ ]:
# 1. Mount Google Drive and set paths (same layout as compare_models.ipynb)
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = "/content/drive/MyDrive/Uskudar Uni Master/YOLO Model klasor"
DATASET_PATH = os.path.join(BASE, "Test Dataset")
FINETUNED_MODEL_PATH = os.path.join(BASE, "runs/bosphorus_yolo11s_10884/weights/best.pt")
OUTPUT_DIR = os.path.join(BASE, "Comparisons", "revision_outputs")

TEST_IMAGES_PATH = os.path.join(DATASET_PATH, "images")
TEST_LABELS_PATH = os.path.join(DATASET_PATH, "labels")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output dir:", OUTPUT_DIR)
print("Test images found:", len([f for f in os.listdir(TEST_IMAGES_PATH) if f.lower().endswith(('.jpg','.jpeg','.png'))]))

In [ ]:
# 2. Install Ultralytics
!pip install -q -U ultralytics

import ultralytics, torch
from ultralytics import YOLO
print("Ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU mode - slower but works)")

In [ ]:
# 3. Helper functions (verbatim from compare_models.ipynb protocol)
import numpy as np
import time
import cv2

def parse_yolo_label(label_path, img_width, img_height):
    # Handles both label formats present in the Roboflow export:
    #   box lines:     class xc yc w h            (5 fields)
    #   polygon lines: class x1 y1 x2 y2 ... xn yn (odd field count > 5)
    # Polygons are converted to their axis-aligned bounding box (min/max hull),
    # matching the behaviour of the original evaluation run.
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                x_center = float(parts[1]) * img_width
                y_center = float(parts[2]) * img_height
                width = float(parts[3]) * img_width
                height = float(parts[4]) * img_height
                boxes.append([x_center - width/2, y_center - height/2,
                              x_center + width/2, y_center + height/2])
            elif len(parts) > 5 and len(parts) % 2 == 1:
                xs = [float(v) * img_width for v in parts[1::2]]
                ys = [float(v) * img_height for v in parts[2::2]]
                boxes.append([min(xs), min(ys), max(xs), max(ys)])
    return boxes

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    return intersection / union if union > 0 else 0

def match_boxes(pred_boxes, gt_boxes, iou_threshold=0.5):
    # Greedy matching, identical logic to compute_detection_metrics in compare_models.ipynb.
    # Returns (tp, fp, fn, matched_ious list).
    if not pred_boxes:
        return 0, 0, len(gt_boxes), []
    if not gt_boxes:
        return 0, len(pred_boxes), 0, []
    gt_matched = [False] * len(gt_boxes)
    matched_ious = []
    tp = 0
    for pred in pred_boxes:
        best_iou = 0; best_gt_idx = -1
        for gt_idx, gt in enumerate(gt_boxes):
            if gt_matched[gt_idx]:
                continue
            iou = calculate_iou(pred, gt)
            if iou > best_iou:
                best_iou = iou; best_gt_idx = gt_idx
        if best_iou >= iou_threshold and best_gt_idx >= 0:
            gt_matched[best_gt_idx] = True
            matched_ious.append(best_iou)
            tp += 1
    fp = len(pred_boxes) - tp
    fn = len(gt_boxes) - tp
    return tp, fp, fn, matched_ious

print("Helpers ready.")

In [ ]:
# 4. Load the five models
MODEL_SPECS = {
    'YOLOv8n':    {'weights': 'yolov8n.pt', 'imgsz': 640},
    'YOLOv8s':    {'weights': 'yolov8s.pt', 'imgsz': 640},
    'YOLO11n':    {'weights': 'yolo11n.pt', 'imgsz': 640},
    'YOLO11s':    {'weights': 'yolo11s.pt', 'imgsz': 640},
    'Fine-tuned': {'weights': FINETUNED_MODEL_PATH, 'imgsz': 1088},
}
models = {}
for name, spec in MODEL_SPECS.items():
    models[name] = YOLO(spec['weights'])
    print("Loaded", name)

In [ ]:
# 5. Run inference twice per model per image:
#    pass A at conf 0.01 (for post-hoc threshold sweep and PR curves)
#    pass B at conf 0.25 (protocol-exact, used for the headline table, grid and IoU sensitivity)
from PIL import Image

test_images = sorted([f for f in os.listdir(TEST_IMAGES_PATH) if f.lower().endswith(('.jpg','.jpeg','.png'))])
print(len(test_images), "test images")

# cache: {model: {img: {'low': (boxes, confs), 'p25': (boxes, confs), 'gt': [...] }}}
cache = {}
gt_cache = {}
for img_name in test_images:
    img_path = os.path.join(TEST_IMAGES_PATH, img_name)
    with Image.open(img_path) as im:
        w, h = im.size
    label_path = os.path.join(TEST_LABELS_PATH, os.path.splitext(img_name)[0] + '.txt')
    gt_cache[img_name] = parse_yolo_label(label_path, w, h)

total_gt = sum(len(v) for v in gt_cache.values())
print("Total ground-truth vessels:", total_gt)

# Verify parsed ground truth against the published evidence from the original run
# (per-image gt counts, evidence/comparison_metrics_60images.csv on GitHub) before spending GPU time.
import urllib.request, csv, io
EV_URL = "https://raw.githubusercontent.com/Mizar-107/Uskudar-University-Thesis-Repository/main/evidence/comparison_metrics_60images.csv"
with urllib.request.urlopen(EV_URL) as r:
    ev_rows = list(csv.DictReader(io.TextIOWrapper(r, encoding='utf-8')))
ev_gt = {row['image']: int(row['gt_count']) for row in ev_rows}
mismatches = [(n, len(gt_cache.get(n, [])), g) for n, g in ev_gt.items() if len(gt_cache.get(n, [])) != g]
if mismatches:
    for n, got, exp in mismatches[:10]:
        print(f"MISMATCH {n}: parsed {got}, evidence says {exp}")
    raise SystemExit("Ground-truth verification failed - stopping before inference.")
print(f"Ground-truth verification passed: {len(ev_gt)} images, counts match the published evidence, total {sum(ev_gt.values())} boxes.")

for name, spec in MODEL_SPECS.items():
    cache[name] = {}
    t0 = time.time()
    for img_name in test_images:
        img_path = os.path.join(TEST_IMAGES_PATH, img_name)
        entry = {}
        for key, conf in (('low', 0.01), ('p25', 0.25)):
            results = models[name].predict(source=img_path, conf=conf, imgsz=spec['imgsz'], classes=None, verbose=False)
            boxes = [box.xyxy[0].cpu().numpy().tolist() for r in results for box in r.boxes]
            confs = [float(box.conf[0]) for r in results for box in r.boxes]
            entry[key] = (boxes, confs)
        cache[name][img_name] = entry
    print(f"{name}: done in {time.time()-t0:.1f}s")

In [ ]:
# 6. Confidence-threshold sweep table
import pandas as pd

def aggregate(model_name, source_key, conf_filter=None, iou_thr=0.5):
    TP = FP = FN = 0
    all_matched = []
    for img_name in test_images:
        boxes, confs = cache[model_name][img_name][source_key]
        if conf_filter is not None:
            boxes = [b for b, c in zip(boxes, confs) if c >= conf_filter]
        tp, fp, fn, mi = match_boxes(boxes, gt_cache[img_name], iou_thr)
        TP += tp; FP += fp; FN += fn; all_matched.extend(mi)
    P = TP / (TP + FP) if TP + FP else 0.0
    R = TP / (TP + FN) if TP + FN else 0.0
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    mIoU = float(np.mean(all_matched)) if all_matched else 0.0
    return {'precision_%': round(P*100, 2), 'recall_%': round(R*100, 2), 'f1_%': round(F1*100, 2),
            'mean_iou': round(mIoU, 3), 'tp': TP, 'fp': FP, 'fn': FN, 'detections': TP + FP}

CONF_POINTS = [0.10, 0.25, 0.40, 0.50]
rows = []
for name in MODEL_SPECS:
    for t in CONF_POINTS:
        if t == 0.25:
            r = aggregate(name, 'p25')            # protocol-exact direct predictions
        else:
            r = aggregate(name, 'low', conf_filter=t)  # post-hoc from the 0.01 pass
        rows.append({'model': name, 'conf': t, **r})
sweep_df = pd.DataFrame(rows)
sweep_df.to_csv(os.path.join(OUTPUT_DIR, 'conf_sweep.csv'), index=False)
print(sweep_df.to_string(index=False))
ft25 = sweep_df[(sweep_df.model == 'Fine-tuned') & (sweep_df.conf == 0.25)].iloc[0]
print()
print(f"Consistency check vs thesis Table 6: recall {ft25['recall_%']}% (expected 87.54), precision {ft25['precision_%']}% (expected 80.06)")

In [ ]:
# 7. IoU matching-threshold sensitivity (at protocol conf 0.25)
rows = []
for name in MODEL_SPECS:
    for iou_t in [0.30, 0.50, 0.70]:
        r = aggregate(name, 'p25', iou_thr=iou_t)
        rows.append({'model': name, 'iou_threshold': iou_t, **r})
iou_df = pd.DataFrame(rows)
iou_df.to_csv(os.path.join(OUTPUT_DIR, 'iou_sensitivity.csv'), index=False)
print(iou_df.to_string(index=False))

In [ ]:
# 8. Precision-recall curves: fine-tuned vs strongest vanilla (YOLO11s)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

curve_points = [round(0.05 * i, 2) for i in range(1, 19)]  # 0.05 .. 0.90
fig, ax = plt.subplots(1, 1, figsize=(7, 5.2), dpi=150)
curve_rows = []
for name, style in [('Fine-tuned', '-o'), ('YOLO11s', '-s')]:
    P, R = [], []
    for t in curve_points:
        r = aggregate(name, 'low', conf_filter=t)
        P.append(r['precision_%']); R.append(r['recall_%'])
        curve_rows.append({'model': name, 'conf': t, 'precision_%': r['precision_%'], 'recall_%': r['recall_%'], 'f1_%': r['f1_%']})
    ax.plot(R, P, style, markersize=3.5, linewidth=1.4, label=name)
ax.set_xlabel('Recall (%)'); ax.set_ylabel('Precision (%)')
ax.set_xlim(0, 100); ax.set_ylim(0, 100)
ax.grid(alpha=0.3); ax.legend()
ax.set_title('Precision-recall behaviour across confidence thresholds (60-image test set)')
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'fig_pr_curves.png'), bbox_inches='tight')
pd.DataFrame(curve_rows).to_csv(os.path.join(OUTPUT_DIR, 'pr_curve_points.csv'), index=False)
plt.show()
print("Saved fig_pr_curves.png and pr_curve_points.csv")

In [ ]:
# 9. Regenerate comparison grid with English labels ("ship")
import random

random.seed(42)
NUM_IMAGES = 8
selected = random.sample(test_images, min(NUM_IMAGES, len(test_images)))

CELL_W = 520
def draw_panel(img_path, boxes, confs, gts, model_name, is_ft):
    img = cv2.imread(img_path)
    tp_flags = []
    gt_matched = [False] * len(gts)
    for b in boxes:
        best_iou = 0; best_idx = -1
        for gi, g in enumerate(gts):
            if gt_matched[gi]:
                continue
            iou = calculate_iou(b, g)
            if iou > best_iou:
                best_iou = iou; best_idx = gi
        ok = best_iou >= 0.5 and best_idx >= 0
        if ok:
            gt_matched[best_idx] = True
        tp_flags.append(ok)
    for b, c, ok in zip(boxes, confs, tp_flags):
        color = (60, 200, 60) if ok else (50, 50, 230)
        x1, y1, x2, y2 = [int(v) for v in b]
        cv2.rectangle(img, (x1, y1), (x2, y2), color, max(2, img.shape[1] // 640))
        label = f"ship {c:.2f}" if is_ft else f"{c:.2f}"
        fs = max(0.5, img.shape[1] / 1600)
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, fs, 2)
        ty = y1 - 6 if y1 - th - 8 > 0 else y2 + th + 6
        cv2.rectangle(img, (x1, ty - th - 4), (x1 + tw + 4, ty + 4), color, -1)
        cv2.putText(img, label, (x1 + 2, ty), cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 255), 2)
    h, w = img.shape[:2]
    scale = CELL_W / w
    return cv2.resize(img, (CELL_W, int(h * scale)))

header_h = 46
cols = list(MODEL_SPECS.keys())
grid_rows = []
for img_name in selected:
    img_path = os.path.join(TEST_IMAGES_PATH, img_name)
    panels = []
    for name in cols:
        boxes, confs = cache[name][img_name]['p25']
        panels.append(draw_panel(img_path, boxes, confs, gt_cache[img_name], name, name == 'Fine-tuned'))
    hmax = max(p.shape[0] for p in panels)
    panels = [cv2.copyMakeBorder(p, 0, hmax - p.shape[0], 0, 4, cv2.BORDER_CONSTANT, value=(255,255,255)) for p in panels]
    grid_rows.append(np.hstack(panels))

header = np.full((header_h, grid_rows[0].shape[1], 3), 255, dtype=np.uint8)
for i, name in enumerate(cols):
    cv2.putText(header, name, (i * (CELL_W + 4) + 14, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.95, (30, 30, 30), 2)
grid = np.vstack([header] + grid_rows)
out_path = os.path.join(OUTPUT_DIR, 'comparison_grid_en.png')
cv2.imwrite(out_path, grid)
print("Saved", out_path, grid.shape)
print("Green = correct detection (IoU >= 0.5), red = false positive. Fine-tuned labels read 'ship'.")

In [ ]:
# 10. Showcase figures: fine-tuned model on the three densest scenes, English labels
dense = sorted(test_images, key=lambda n: len(gt_cache[n]), reverse=True)[:3]
for i, img_name in enumerate(dense, 1):
    img_path = os.path.join(TEST_IMAGES_PATH, img_name)
    boxes, confs = cache['Fine-tuned'][img_name]['p25']
    panel = draw_panel(img_path, boxes, confs, gt_cache[img_name], 'Fine-tuned', True)
    out = os.path.join(OUTPUT_DIR, f'fig_showcase_{i}.png')
    cv2.imwrite(out, panel)
    print(f"Saved {out} ({img_name}: {len(gt_cache[img_name])} ground-truth vessels, {len(boxes)} detections)")

In [ ]:
# 11. Dataset statistics from the published export (bogaz_v_1)
import glob as _glob

CANDIDATES = [
    os.path.join(BASE, "Custom Dataset", "bogaz_v_1.v3i.yolov8"),
    "/content/drive/MyDrive/Uskudar Uni Master/YOLO Model klasor/Custom Dataset/bogaz_v_1.v3i.yolov8",
]
ds_root = next((p for p in CANDIDATES if os.path.isdir(p)), None)
if ds_root is None:
    for root, dirs, files in os.walk("/content/drive/MyDrive"):
        for d in dirs:
            if d.startswith("bogaz_v_1"):
                ds_root = os.path.join(root, d); break
        if ds_root:
            break
print("Dataset root:", ds_root)

stats = {}
all_counts, all_areas, all_ars, centers = [], [], [], []
for split in ['train', 'valid', 'test']:
    lbl_dir = os.path.join(ds_root, split, 'labels')
    img_dir = os.path.join(ds_root, split, 'images')
    if not os.path.isdir(lbl_dir):
        continue
    lbl_files = sorted(_glob.glob(os.path.join(lbl_dir, '*.txt')))
    n_imgs = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]) if os.path.isdir(img_dir) else len(lbl_files)
    n_boxes = 0
    for lf in lbl_files:
        cnt = 0
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cnt += 1
                    wn, hn = float(parts[3]), float(parts[4])
                    cx, cy = float(parts[1]), float(parts[2])
                elif len(parts) > 5 and len(parts) % 2 == 1:
                    cnt += 1
                    xs = [float(v) for v in parts[1::2]]
                    ys = [float(v) for v in parts[2::2]]
                    wn, hn = max(xs) - min(xs), max(ys) - min(ys)
                    cx, cy = (max(xs) + min(xs)) / 2, (max(ys) + min(ys)) / 2
                else:
                    continue
                all_areas.append(wn * hn)
                all_ars.append(wn / hn if hn > 0 else 0)
                centers.append((cx, cy))
        n_boxes += cnt
        all_counts.append(cnt)
    stats[split] = {'images': n_imgs, 'labels': len(lbl_files), 'boxes': n_boxes}
    print(f"{split}: {n_imgs} images, {n_boxes} boxes")

total_imgs = sum(s['images'] for s in stats.values())
total_boxes = sum(s['boxes'] for s in stats.values())
print(f"TOTAL: {total_imgs} images, {total_boxes} boxes")

fig, axes = plt.subplots(2, 2, figsize=(11, 8), dpi=140)
axes[0,0].hist(all_counts, bins=range(0, max(all_counts)+2), color='#2b6f8c', edgecolor='white')
axes[0,0].set_title('Annotated vessels per image'); axes[0,0].set_xlabel('boxes'); axes[0,0].set_ylabel('images')
axes[0,1].hist([a*100 for a in all_areas], bins=40, color='#2b6f8c', edgecolor='white', log=True)
axes[0,1].set_title('Bounding-box area (% of image, log scale)'); axes[0,1].set_xlabel('% of image area')
axes[1,0].hist(all_ars, bins=40, range=(0, 8), color='#2b6f8c', edgecolor='white')
axes[1,0].set_title('Box aspect ratio (width/height)'); axes[1,0].set_xlabel('aspect ratio')
hx = [c[0] for c in centers]; hy = [c[1] for c in centers]
hb = axes[1,1].hist2d(hx, hy, bins=40, cmap='viridis')
axes[1,1].set_title('Box-centre spatial density'); axes[1,1].invert_yaxis()
fig.colorbar(hb[3], ax=axes[1,1])
fig.suptitle('Dataset statistics: bogaz_v_1 (all splits)', y=1.0)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'fig_dataset_stats.png'), bbox_inches='tight')
plt.show()

import json as _json
summary = {'splits': stats, 'total_images': total_imgs, 'total_boxes': total_boxes,
           'boxes_per_image_mean': float(np.mean(all_counts)), 'boxes_per_image_median': float(np.median(all_counts)),
           'boxes_per_image_max': int(max(all_counts)), 'box_area_pct_median': float(np.median(all_areas))*100,
           'box_area_pct_mean': float(np.mean(all_areas))*100, 'aspect_ratio_median': float(np.median(all_ars))}
with open(os.path.join(OUTPUT_DIR, 'dataset_stats.json'), 'w') as f:
    _json.dump(summary, f, indent=2)
print(_json.dumps(summary, indent=2))

In [ ]:
# 12. Package everything
import shutil
zip_path = os.path.join(OUTPUT_DIR, '..', 'revision_outputs')
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print("Zipped to", zip_path + ".zip")
print()
print("Files in", OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    print("  -", f, f"({os.path.getsize(os.path.join(OUTPUT_DIR, f))//1024} KB)")
print()
print("Done. All outputs are in Google Drive under Comparisons/revision_outputs/.")